# Finn emne som brukar New Quizzes

For å varsle om at vi legg om til desimalkomma treng eg alle dei som i dette semesteret har New Quizzes innebygd.

In [35]:
import requests
import os
import pandas as pd
tokenCanvas = os.environ['tokenCanvas']

headers = {
    'Authorization': f'Bearer {tokenCanvas}',
}
params = {
    'per_page': 100,
    'show_inherited': True
}


In [36]:

def finn_rel(link_header):
    link_header_dict = {}
    for link in link_header.split(","):
        url, rel = link.strip().split(";")
        rel = rel.split('=')[1]
        link_header_dict[rel.strip().replace('"', '')] = url.strip().replace('<', '').replace('>', '')      # Kommentar: denne er med BERRE for å skape balanse i .org-fila: \"
    return link_header_dict

Først henter eg *alle* emne ved HVL som er aktive dette semesteret: 

In [37]:
    
url = "https://hvl.instructure.com/api/v1/accounts/1/courses"   # Endre til aktuelt endepunkt
dr_liste = []
hentmeir = True
terminar = [206, 216, 217, 224, 227, 228, 333, 334, 335, 336, 337, 338, 339, 349, 352, 353, 357, 358, 365]
while hentmeir:
    respons = requests.get(url, headers=headers, params=params)
    if 200 <= respons.status_code < 300:
        data = respons.json()
        hentmeir = "next" in respons.headers['link']
        if hentmeir:
            url = finn_rel(respons.headers['link'])['next']
            print(url)   # for kontrollen sin skuld
# Hent ut data (NB! dette vil variere frå endepunkt til endepunkt)
        dataliste = []
        for element in data:
            if element['enrollment_term_id'] in terminar:   # Filtrering på terminer    
                dataliste.append([element['id'], element['name']])
        df = pd.DataFrame(dataliste, columns=['id', 'name'])
        dr_liste.append(df)
    # hentmeir = False
# og etter at eg er ferdig å lese inn slår eg dei saman:
alledata = pd.concat((df for df in dr_liste if not df.empty), ignore_index=True)

https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=2&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=3&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=4&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=5&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=6&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=7&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=8&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=9&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=10&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show_inherited=True&page=11&per_page=100
https://hvl.instructure.com/api/v1/accounts/1/courses?show

In [38]:
aktuelle_emne = pd.DataFrame(alledata)
aktuelle_emne.to_csv('aktuelle_emne.csv', index=False)

Så går eg gjennom alle og sjekker om dei har "New Quizzes":

In [41]:
emne_med_quizer = []
for r, c in  aktuelle_emne.iterrows():
    url = f"https://hvl.instructure.com/api/quiz/v1/courses/{c['id']}/quizzes"
    respons = requests.get(url, headers=headers, params=params)
    if 200 <= respons.status_code < 300:
        data = respons.json()
        if data:
            emne_med_quizer.append([c['id'], c['name']])
            print(f"{c['id']}: {c['name']}")

30032: Kull 85D: SYKDF120P Praksisstudie, grunnleggande sjukepleie vår – haust 25
31320: BØA111 Bergen - Matematikk for økonomer  (1)
31324: BØA111 Sogndal - Matematikk for økonomer (3)
31394: BØA119 Haugesund - Danning og akademisk håndverk  (1)
31399: BØA119 Sogndal - Danning og akademisk håndverk (2)
31401: BØA119 Bergen - Danning og akademisk håndverk (3)
31443: MGBEN/MGUEN101 Stord (2) - H25
31484: MGUMAT302 Bergen  (1)
31588: MAT101 Diskret matematikk for Informasjonsteknologi, Haugesund høsten 2025
31610: MAT110 Matematikk 1 - Bergen Maskin- og marinfag H25
31621: MAT110 Førde - Matematikk 1 hausten 2025
31630: IDF100 25H Danning og akademisk handverk
31634: MAT110 Haugesund - Matematikk 1 høsten 2025
31652: MAT110 Bergen - Matematikk 1 for Bygg høsten 2025
31664: MAT110 Bergen - Matematikk 1 for Elektroteknologi og Kjemi høsten 2025
31922: VPDP200 Sogndal og Haugesund 25H
31944: Bergen VPL110 (1)
31948: Sogndal VPL110 (2)
31981: SAB110 Bergen, Høst 25 (1)
31983: SAB110 Sogndal,

In [42]:
emne_med_quizer_df = pd.DataFrame(emne_med_quizer, columns=['id', 'name'])

Så henter eg ut alle lærarane i dei emna:

In [43]:
temp = []
for r, c in emne_med_quizer_df.iterrows():
    url = f"https://hvl.instructure.com/api/v1/courses/{c['id']}/enrollments"
    params = {
        'type[]': ['TeacherEnrollment'],
        'per_page': 100,
        'type[]': 'TeacherEnrollment'
    }
    respons = requests.get(url, headers=headers, params=params)
    if 200 <= respons.status_code < 300:
        data = respons.json()
        lærarar = []
        for p in data:
            if p['type'] == 'TeacherEnrollment':
                url = f"https://hvl.instructure.com/api/v1/users/{p['user_id']}"
                respons = requests.get(url, headers=headers)
                if 200 <= respons.status_code < 300:
                    brukerdata = respons.json()
                    epost = brukerdata['email']
                lærarar.append(epost)
        temp.append([c['id'], c['name'], list(set(lærarar))])
        print(f"{c['id']}: {len(list(set(lærarar)))}")
        

30032: 10
31320: 4
31324: 4
31394: 2
31399: 1
31401: 2
31443: 2
31484: 7
31588: 3
31610: 7
31621: 6
31630: 9
31634: 3
31652: 3
31664: 10
31922: 11
31944: 5
31948: 4
31981: 6
31983: 1
32001: 1
32022: 7
32109: 4
32124: 25
32163: 4
32167: 3
32175: 7
32177: 10
32184: 3
32206: 1
32209: 1
32320: 7
32434: 2
32448: 2
32476: 12
32489: 10
32492: 4
32504: 1
32560: 2
32633: 1
32635: 7
32686: 1
32982: 2
33021: 2
33038: 7
33059: 11
33130: 10
33175: 8
33199: 6
33203: 4
33211: 5
33214: 6
33220: 19
33227: 14
33228: 6
33230: 6
33282: 7
33285: 9
33292: 12
33300: 13
33310: 1
33315: 7
33319: 8
33321: 2
33373: 2
33374: 10
33375: 1
33382: 19
33384: 1
33399: 12
33472: 10
33490: 14
33492: 15
33494: 15
33511: 1
33648: 2
33658: 4
34024: 4
34026: 4
34029: 5
34102: 11


Til slutt står eg att med ei liste på 81 emne:

In [46]:
pd.DataFrame(temp, columns=['id', 'name', 'lærarar']).to_csv('emne_med_quizer_og_lærarar.csv', index=False)

Denne lista går eg så gjennom og plukker ut dei emna som bruker New Quizzes til "matematiske" spørsmål. Altså utelukker eg alle språkfag, danningsemna og tilsvarande. Denne lista (åtte emne) går eg så gjennom i eit anna program.